In [2]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [38]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/crop_recommendation_merged.csv")
df.head()


,Soil_Type,Soil_pH,Moisture_Level,Season,Location_Region,Previous_Crop,Recommended_Crop
0,Loamy,4.57,Low,Zaid,Tamil Nadu,Rice,Banana
1,Laterite,6.88,High,Winter,Karnataka,Wheat,Rice
2,Clay,7.15,High,Rabi,Uttar Pradesh,NaN,Cotton
3,Laterite,5.89,Low,Kharif,Karnataka,Groundnut,Cumin
4,Alluvial,6.76,High,Rabi,Gujarat,Sugarcane,Pulses


In [39]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Soil_Type         10000 non-null  object 
 1   Soil_pH           10000 non-null  float64
 2   Moisture_Level    10000 non-null  object 
 3   Season            10000 non-null  object 
 4   Location_Region   10000 non-null  object 
 5   Previous_Crop     8776 non-null   object 
 6   Recommended_Crop  10000 non-null  object 
dtypes: float64(1), object(6)
memory usage: 547.0+ KB


In [40]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
df.columns


Index(['soil_type', 'soil_ph', 'moisture_level', 'season', 'location_region',
       'previous_crop', 'recommended_crop'],
      dtype='object')

In [41]:
df.isna().sum()

,0
soil_type,0
soil_ph,0
moisture_level,0
season,0
location_region,0
previous_crop,1224
recommended_crop,0


In [43]:
df["previous_crop"] = df["previous_crop"].fillna("None")


In [44]:
df.isna().sum()

,0
soil_type,0
soil_ph,0
moisture_level,0
season,0
location_region,0
previous_crop,0
recommended_crop,0


In [45]:
df = df.drop(columns=["location_region"])

In [46]:
df["recommended_crop"].value_counts()


,count
recommended_crop,
Maize,1054
Pulses,1014
Rice,1011
Cotton,1010
Sugarcane,1007
Banana,1004
Soybean,1001
Cumin,982
Wheat,965


In [47]:
df["ph_category"] = pd.cut(
    df["soil_ph"],
    bins=[0, 5.5, 7.5, 14],
    labels=["acidic", "neutral", "alkaline"]
)

df[["soil_ph", "ph_category"]].head()


,soil_ph,ph_category
0,4.57,acidic
1,6.88,neutral
2,7.15,neutral
3,5.89,neutral
4,6.76,neutral


In [48]:
X = df.drop("recommended_crop", axis=1)
y = df["recommended_crop"]

X.head(), y.head()


(  soil_type  soil_ph moisture_level  season previous_crop ph_category
 0     Loamy     4.57            Low    Zaid          Rice      acidic
 1  Laterite     6.88           High  Winter         Wheat     neutral
 2      Clay     7.15           High    Rabi          None     neutral
 3  Laterite     5.89            Low  Kharif     Groundnut     neutral
 4  Alluvial     6.76           High    Rabi     Sugarcane     neutral,
 0    Banana
 1      Rice
 2    Cotton
 3     Cumin
 4    Pulses
 Name: recommended_crop, dtype: object)

In [50]:
X.head()
# y.head()

,soil_type,soil_ph,moisture_level,season,previous_crop,ph_category
0,Loamy,4.57,Low,Zaid,Rice,acidic
1,Laterite,6.88,High,Winter,Wheat,neutral
2,Clay,7.15,High,Rabi,None,neutral
3,Laterite,5.89,Low,Kharif,Groundnut,neutral
4,Alluvial,6.76,High,Rabi,Sugarcane,neutral


In [51]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Check mapping (important)
dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))


{'Banana': 0,
 'Cotton': 1,
 'Cumin': 2,
 'Groundnut': 3,
 'Maize': 4,
 'Pulses': 5,
 'Rice': 6,
 'Soybean': 7,
 'Sugarcane': 8,
 'Wheat': 9}

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)

X_train.shape, X_test.shape


((7500, 6), (2500, 6))

In [53]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

cat_cols, num_cols


(['soil_type', 'moisture_level', 'season', 'previous_crop'], ['soil_ph'])

In [17]:
cat_cols.append("ph_category")


In [56]:
from xgboost import XGBClassifier
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss"
)


In [58]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Re-identify columns AFTER dropping location_region
cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Rebuild preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# Rebuild pipeline
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ]
)

# Train again
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['soil_type',
                                                   'moisture_level', 'season',
                                                   'previous_crop']),
                                                 ('num', 'passthrough',
                                                  ['soil_ph'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.9, device=None,
                               ear...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=600, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [36]:
y_pred = pipeline.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.0992


In [60]:
import numpy as np

# Predict probabilities
probs = pipeline.predict_proba(X_test)

# Get indices of top 3 probabilities
top3_idx = np.argsort(probs, axis=1)[:, -3:]

# Get class labels
classes = pipeline.named_steps["model"].classes_

# Convert indices to labels
top3_labels = classes[top3_idx]

# Compute Top-3 accuracy
top3_accuracy = np.mean([
    y_test[i] in top3_labels[i]
    for i in range(len(y_test))
])

print("Top-3 Accuracy:", top3_accuracy)


Top-3 Accuracy: 0.3012


In [71]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Final feature list (STRICT)
categorical_features = [
    "soil_type",
    "moisture_level",
    "season",
    "previous_crop"
]

numeric_features = ["soil_ph"]

# Categorical preprocessing
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=10  # 🚀 reduces noise, BIG accuracy gain
        ))
    ]
)

# Numeric preprocessing
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipeline, categorical_features),
        ("num", num_pipeline, numeric_features)
    ]
)


In [72]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_train)),
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.2,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)


In [74]:
# Normalize column names (CRITICAL FIX)
df.columns = (
    df.columns
    .str.strip()          # remove leading/trailing spaces
    .str.lower()          # lowercase everything
    .str.replace(" ", "_")
)

print(df.columns.tolist())


['soil_type', 'soil_ph', 'moisture_level', 'season', 'previous_crop', 'recommended_crop', 'ph_category']


In [75]:
import numpy as np
from sklearn.pipeline import Pipeline

# Rebuild pipeline with updated model
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ]
)

# Train
pipeline.fit(X_train, y_train)

# Predict probabilities
probs = pipeline.predict_proba(X_test)

# Get top-3 class indices
top3_idx = np.argsort(probs, axis=1)[:, -3:]

# Map indices to crop labels
class_labels = pipeline.named_steps["model"].classes_
top3_preds = class_labels[top3_idx]

# Top-3 accuracy
top3_accuracy = np.mean([
    y_test[i] in top3_preds[i]
    for i in range(len(y_test))
])

print("✅ Top-3 Accuracy:", round(top3_accuracy, 4))


✅ Top-3 Accuracy: 0.3064


In [76]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']